# `TransformersPipeline` model wrapper

If you're using a standard Hugging Face pipeline, you might notice they often return lists of dictionaries—which is great for humans, but not so great for SHAP explainers that expect numbers and arrays.

The `shap.models.TransformersPipeline` wrapper bridge this gap. It turns those dictionary outputs into neat NumPy arrays that SHAP can use to calculate feature importance.

In [1]:
import shap
import transformers
import numpy as np

# Let's create a standard sentiment-analysis pipeline.
classifier = transformers.pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

print(f"Standard pipeline output: {classifier(['I love working with SHAP!'])}")

## Wrapping the pipeline

Once we wrap the classifier, it starts returning an array where each column is a label score.

In [2]:
shap_pipeline = shap.models.TransformersPipeline(classifier)

my_text = ["I love working with SHAP!", "I'm not a fan of bugs."]
scores = shap_pipeline(my_text)

print(f"Scores array shape: {scores.shape}")
print(f"Scores for the first sentence: {scores[0]}")
print(f"Labels the model knows: {shap_pipeline.id2label}")

## Converting to Logits

A lot of SHAP explainers work better if you give them 'logits' (raw scores before they are squeezed into probabilities). You can tell the wrapper to handle this conversion automatically.

In [3]:
logit_pipeline = shap.models.TransformersPipeline(classifier, rescale_to_logits=True)
logit_scores = logit_pipeline(my_text)

print(f"Logit-based scores for the first sentence: {logit_scores[0]}")